# Medvlm-Image-Reliance on Colab

Runtime > Change runtime type > **GPU**. Add a Colab secret named `HF_TOKEN` (key icon, left bar) from an account that has accepted the MedGemma terms.
All state lives in git (manifests) and Drive (raw outputs), so a disconnect only costs the current chunk: just re-run the cells.

In [9]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['MEDVLM_OUTPUTS_DIR'] = '/content/drive/MyDrive/medvlm_outputs'   # survives disconnects
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.makedirs(os.environ['MEDVLM_OUTPUTS_DIR'], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
REPO = 'https://github.com/thaitri2005/Medvlm-Image-Reliance.git'
if not os.path.exists('/content/Medvlm-Image-Reliance'):
    !git clone $REPO /content/Medvlm-Image-Reliance
%cd /content/Medvlm-Image-Reliance
!git pull --ff-only
!pip install -q -e ".[model,gpu,dev]"

/content/Medvlm-Image-Reliance
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 4), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 3.38 KiB | 1.13 MiB/s, done.
From https://github.com/thaitri2005/Medvlm-Image-Reliance
   24458b2..f954356  main       -> origin/main
Updating 24458b2..f954356
Fast-forward
 configs/models/gemma3_4b.yaml  |   1 +
 configs/models/medgemma4b.yaml |   3 +
 notebooks/colab_run.ipynb      | 412 +++++++++++++++++++++++++----------------
 3 files changed, 258 insertions(+), 158 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for medvlm (pyproject.toml) ... done


In [11]:
PROFILE, MODEL = 'pilot', 'medgemma4b'
!medvlm doctor --model $MODEL

python      3.13.15 on Linux
torch       2.11.0+cu130  cuda=True
gpu         Tesla T4  14.6 GiB  bf16=True
transformers 5.18.0  (Gemma 3 / MedGemma needs >= 4.50)
hf access   OK for google/medgemma-4b-it
data dir    /content/Medvlm-Image-Reliance/data
outputs dir /content/drive/MyDrive/medvlm_outputs


## Gate: real-model integration tests (do this before any long run)
If these fail on MedGemma, fix the backend first (dtype, processor, padding). On a T4 try `--set model.dtype=float16` or `--set model.quantization=4bit`.

In [12]:
!MEDVLM_TEST_MODEL=$MODEL pytest -m slow -x

....                                                                     [100%]
4 passed, 37 deselected in 78.96s (0:01:18)


In [13]:
!medvlm prepare --profile $PROFILE --model $MODEL

README.md: 100% 4.29k/4.29k [00:00<00:00, 1.36MB/s]

data/train-00000-of-00007-f2d0e9ef9f022d(…): downloading bytes:  73% 31.4M/42.8M [00:01<00:00, 45.7MB/s, 1.79MB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): downloading bytes:  97% 41.5M/42.8M [00:01<00:00, 57.1MB/s, 3.02MB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  12% 5.15M/42.8M [00:01<00:09, 3.97MB/s,  342kB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  15% 6.45M/42.8M [00:01<00:07, 5.07MB/s,  493kB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  19% 8.03M/42.8M [00:01<00:05, 5.80MB/s,  668kB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  25% 10.6M/42.8M [00:01<00:03, 8.94MB/s,  752kB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  31% 13.3M/42.8M [00:01<00:02, 12.2MB/s,  992kB/s  ]
data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:  37% 15.7M/42.8M [00:02<00:02, 12.2MB/s, 1.38MB/s  ]

## Paraphrases (first time only; the result is committed to git)

In [14]:
!medvlm paraphrase --profile $PROFILE --model $MODEL
!medvlm audit-sheet --profile $PROFILE --model $MODEL -n 100

config.json: 100% 661/661 [00:00<00:00, 3.38MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 16.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 13.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 12.4MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 21.4MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 42.4MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  62% 3.84G/6.17G [00:50<00:20, 112MB/s, 84.2MB/s  ] 
Reconstructing (incomplete total...):  67% 4.15G/6.17G [00:52<00:15, 133MB/s,  112MB/s  ]
Reconstructing (incomplete total...):  70% 4.32G/6.17G [01:00<00:27, 66.9MB/s,  133MB/s  ]
Reconstructing (incomplete total...):  76% 4.67G/6.17G [01:00<00:15, 98.7MB/s, 66.9MB/s  ]

Reconstructing 

## Run (resumable; first measure throughput with `--limit 200`)

In [15]:
!medvlm run --profile $PROFILE --model $MODEL --limit 200

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 883/883 [00:28<00:00, 31.05it/s] 
[run] medgemma4b/pilot: 1510 jobs, 0 already done, 200 to run
100% 200/200 [26:07<00:00,  7.84s/job]


In [16]:
!medvlm run --profile $PROFILE --model $MODEL

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 883/883 [00:29<00:00, 30.30it/s] 
[run] medgemma4b/pilot: 1510 jobs, 200 already done, 1310 to run
100% 1310/1310 [1:38:54<00:00,  4.53s/job]


In [17]:
!medvlm analyze --profile $PROFILE --model $MODEL
import pandas as pd
pd.read_csv(f'results/{MODEL}/{PROFILE}/by_dataset.csv').round(3)

[analyze] 330 samples -> /content/Medvlm-Image-Reliance/results/medgemma4b/pilot


,dataset,split,n,n_images,yn_mass_orig,yn_mass_orig_lo,yn_mass_orig_hi,yn_mass_orig_n,correct_orig,correct_orig_lo,...,q_fragile_hi,q_fragile_n,q_dangerous,q_dangerous_lo,q_dangerous_hi,q_dangerous_n,q_worst,q_worst_lo,q_worst_hi,q_worst_n
0,pathvqa,test,90,84,0.995,0.991,0.999,45,0.422,0.315,...,0.319,69,0.246,0.155,0.353,69,0.043,0.000,0.101,69
1,slake,test,150,78,1.000,1.000,1.000,75,0.727,0.653,...,0.240,131,0.366,0.290,0.443,131,0.038,0.008,0.071,131
2,vqa_rad,test,90,75,1.000,1.000,1.000,45,0.667,0.567,...,0.268,71,0.366,0.250,0.479,71,0.141,0.068,0.222,71


## Save what git should keep
Paraphrases and result tables are small. Download them (or push with a GitHub token stored as a Colab secret `GH_TOKEN`) and commit from your laptop.

In [18]:
!zip -qr /content/drive/MyDrive/medvlm_keep.zip manifests results
print('saved to Drive: medvlm_keep.zip')

saved to Drive: medvlm_keep.zip


## Push results to GitHub (replaces the manual Drive download)
One-time setup: GitHub > Settings > Developer settings > Fine-grained tokens > new token limited to this repo with **Contents: read and write**. Add it as a Colab secret named `GH_TOKEN`. Never paste it into a cell.

In [ ]:
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
!git config user.name "thaitri2005"
!git config user.email "110210159+thaitri2005@users.noreply.github.com"
!git add manifests results
!mkdir -p outputs/$MODEL && cp -r "$MEDVLM_OUTPUTS_DIR/$MODEL/$PROFILE" outputs/$MODEL/   # raw outputs live on Drive
!git add -f outputs/$MODEL/$PROFILE   # small enough to keep in git
!git commit -q -m "Results: $MODEL $PROFILE (from Colab)" || echo 'nothing new to commit'
!git pull -q --rebase https://$tok@github.com/thaitri2005/Medvlm-Image-Reliance.git main
!git push -q https://$tok@github.com/thaitri2005/Medvlm-Image-Reliance.git HEAD:main